### Supplemental code for lesson 1

In [2]:
# read from csv
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
print(df.head())
print(df.shape)
print(df.info())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
(891, 15)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null 

In [3]:
# read from api
import requests
import pandas as pd
url = "https://api.open-meteo.com/v1/forecast?latitude=25.76&longitude=-80.19&hourly=temperature_2m"
response = requests.get(url)
data = response.json()
temps = data["hourly"]["temperature_2m"]
df = pd.DataFrame(temps, columns=["temperature"])
df.head()

,temperature
0,21.5
1,21.4
2,21.8
3,22.1
4,22.3



### Kaggle local setup
```bash
# create virtual environment
python -m venv venv 

# activate it (unix based)
source venv/bin/activate

pip install pandas kaggle

# api token generated from kaggle website
export KAGGLE_API_TOKEN=<your_token>

# if configured properly you should see a list of comps
kaggle competitions list
```

In [4]:
def load_kaggle_dataset(dataset_slug, csv_name):
    import os
    import pandas as pd

    !kaggle datasets download -d {dataset_slug}
    !unzip -o *.zip

    return pd.read_csv(csv_name)

df = load_kaggle_dataset(
    "yasserh/titanic-dataset",
    "Titanic-Dataset.csv"
)
df.head()

Dataset URL: https://www.kaggle.com/datasets/yasserh/titanic-dataset
License(s): CC0-1.0
100%|██████████████████████████████████████| 22.0k/22.0k [00:00<00:00, 15.5MB/s]

Archive:  titanic-dataset.zip
  inflating: Titanic-Dataset.csv     


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# website to scrape
url = "https://news.ycombinator.com"

# download page
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
titles = soup.select(".titleline")

data = []

for t in titles:
    data.append(t.text)

df = pd.DataFrame(data, columns=["title"])

df.head()

,title
0,Canada's bill C-22 mandates mass metadata surv...
1,Chrome DevTools MCP (chrome.com)
2,The 49MB web page (thatshubham.com)
3,What Is Agentic Engineering? (simonwillison.net)
4,LLMs can be exhausting (tomjohnell.com)


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

url = "https://news.ycombinator.com"
response = requests.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

story_rows = soup.select("tr.athing")

data = []

for row in story_rows:
    title_tag = row.select_one(".titleline > a")
    site_tag = row.select_one(".sitestr")

    subtext_row = row.find_next_sibling("tr")
    subtext = subtext_row.select_one(".subtext") if subtext_row else None

    score_tag = subtext.select_one(".score") if subtext else None
    author_tag = subtext.select_one(".hnuser") if subtext else None
    age_tag = subtext.select_one(".age") if subtext else None

    comment_tag = None
    if subtext:
        links = subtext.select("a")
        if links:
            comment_tag = links[-1]

    data.append({
        "rank": row.select_one(".rank").get_text(strip=True).replace(".", "") if row.select_one(".rank") else None,
        "title": title_tag.get_text(strip=True) if title_tag else None,
        "article_url": title_tag["href"] if title_tag and title_tag.has_attr("href") else None,
        "site": site_tag.get_text(strip=True) if site_tag else None,
        "points": int(score_tag.get_text(strip=True).replace(" points", "").replace(" point", "")) if score_tag else 0,
        "author": author_tag.get_text(strip=True) if author_tag else None,
        "age": age_tag.get_text(strip=True) if age_tag else None,
        "comments": comment_tag.get_text(strip=True) if comment_tag else None,
        "hn_item_id": row.get("id"),
        "hn_comments_url": urljoin(url, f"item?id={row.get('id')}") if row.get("id") else None,
    })

df = pd.DataFrame(data)
df.head(10)

,rank,title,article_url,site,points,author,age,comments,hn_item_id,hn_comments_url
0,1,Canada's bill C-22 mandates mass metadata surv...,https://www.michaelgeist.ca/2026/03/a-tale-of-...,michaelgeist.ca,412,opengrass,5 hours ago,109 comments,47392084,https://news.ycombinator.com/item?id=47392084
1,2,Chrome DevTools MCP,https://developer.chrome.com/blog/chrome-devto...,chrome.com,377,xnx,7 hours ago,160 comments,47390817,https://news.ycombinator.com/item?id=47390817
2,3,The 49MB web page,https://thatshubham.com/blog/news-audit,thatshubham.com,338,kermatt,7 hours ago,180 comments,47390945,https://news.ycombinator.com/item?id=47390945
3,4,What Is Agentic Engineering?,https://simonwillison.net/guides/agentic-engin...,simonwillison.net,43,lumpa,1 hour ago,29 comments,47393908,https://news.ycombinator.com/item?id=47393908
4,5,LLMs can be exhausting,https://tomjohnell.com/llms-can-be-absolutely-...,tomjohnell.com,96,tjohnell,4 hours ago,77 comments,47391803,https://news.ycombinator.com/item?id=47391803
5,6,LLM Architecture Gallery,https://sebastianraschka.com/llm-architecture-...,sebastianraschka.com,268,tzury,9 hours ago,20 comments,47388676,https://news.ycombinator.com/item?id=47388676
6,7,A new Bigfoot documentary helps explain our co...,https://www.msn.com/en-us/news/us/a-new-bigfoo...,msn.com,52,zdw,4 hours ago,24 comments,47392547,https://news.ycombinator.com/item?id=47392547
7,8,The Linux Programming Interface as a universit...,https://man7.org/tlpi/academic/index.html,man7.org,33,teleforce,3 hours ago,1 comment,47393388,https://news.ycombinator.com/item?id=47393388
8,9,//go:fix inline and the source-level inliner,https://go.dev/blog/inliner,go.dev,118,commotionfever,8 hours ago,43 comments,47339463,https://news.ycombinator.com/item?id=47339463
9,10,Separating the Wayland compositor and window m...,https://isaacfreund.com/blog/river-window-mana...,isaacfreund.com,236,dpassens,11 hours ago,112 comments,47388137,https://news.ycombinator.com/item?id=47388137
